## 1. Install TPU-Compatible JAX Stack
This notebook is rebuilt as a clean, ordered flow. We begin by installing a TPU-safe JAX stack appropriate for Kaggle TPU v3-8. We use JAX 0.4.34 TPU wheels and align NumPy and ml-dtypes. We defer MaxText deps to after clone.

Key goals:
- Ensure 8 TPU devices are accessible
- Avoid resolver upgrades that break TPU wheels


In [10]:
# 1) Install TPU-safe JAX stack
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet

import jax
print("JAX:", jax.__version__, "TPU devices:", jax.device_count())


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
JAX: 0.4.34 TPU devices: 8


## 2. Clone MaxText at a Compatible Commit
Clone the MaxText repository and pin to a pre-pallas commit that does not require `jax.experimental.pallas.ops.attention`. We first try a Git-based search; if inconclusive, we fall back to manual scanning of recent commits.


In [11]:
# 2) Clone and pin MaxText
!git clone https://github.com/google/maxtext.git || true
%cd /kaggle/working/maxtext

import subprocess, os

# Try to find introduction commit for pallas.ops.attention and checkout its parent
patterns = [
    "pallas.ops.attention",
    "from jax.experimental.pallas.ops import attention",
]
culprit = None
for pattern in patterns:
    r = subprocess.run(['git', 'log', '-S', pattern, '--pretty=format:%H', '-n', '1'], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        culprit = r.stdout.strip().split('\n')[0]
        break

if culprit:
    print("First commit with pallas.ops.attention:", culprit)
    subprocess.run(['git', 'checkout', f'{culprit}^'], check=False)
    print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
else:
    # Fallback: pick a known pre-pallas date range (e.g., <= 2024-03-15) and pick that commit
    fallback = subprocess.check_output(['git', 'rev-list', '-n', '1', '--before=2024-03-15', 'HEAD'], text=True).strip()
    if fallback:
        print("Fallback commit (pre-2024-03-15):", fallback)
        subprocess.run(['git', 'checkout', fallback], check=False)
        print(subprocess.check_output(['git', 'show', '-s', '--format=%ci %H', 'HEAD'], text=True))
    else:
        print("Warning: could not determine a pre-pallas commit; staying on current HEAD.")

# Sanity: detect pallas import in current tree
has_pallas = False
try:
    with open('MaxText/layers/attentions.py', 'r') as f:
        has_pallas = 'pallas.ops.attention' in f.read()
except FileNotFoundError:
    pass
print("attentions.py uses pallas:", has_pallas)


Cloning into 'maxtext'...
remote: Enumerating objects: 55709, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 55709 (delta 89), reused 80 (delta 54), pack-reused 55557 (from 2)
Receiving objects: 100% (55709/55709), 317.28 MiB | 29.79 MiB/s, done.
Resolving deltas: 100% (41346/41346), done.


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working/maxtext
First commit with pallas.ops.attention: 58da4567edab27d3fc83600b925219f3d89d5b0b


Previous HEAD position was 5a6580f3 add grain instructions
HEAD is now at c7af09f5 Merge pull request #290 from google:yooh-fix-imports


2023-12-13 22:36:49 -0800 c7af09f53504e19ee92a8d066d8524e8e1ca33c7

attentions.py uses pallas: False


In [12]:
# Find and remove all __pycache__ directories to prevent using stale code
!find . -type d -name "__pycache__" -exec rm -r {} +
print("✅ Python bytecode cache cleared successfully.")


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


✅ Python bytecode cache cleared successfully.


## 3. Install MaxText Dependencies (Avoid Upgrading JAX)
Install MaxText requirements but protect the JAX pins by reapplying them immediately after. This balances repo requirements with TPU-safe versions.


In [14]:
# 3) Install MaxText deps, then re-pin JAX stack
%cd /kaggle/working/maxtext
!pip install -r requirements.txt --quiet || true

# Re-assert JAX pins to prevent resolver upgrades breaking TPU wheels
!pip install --no-deps --force-reinstall \
  "numpy==1.26.4" \
  "ml-dtypes==0.4.0" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jaxlib==0.4.34" \
  --quiet
!pip install --no-deps --force-reinstall \
  "jax[tpu]==0.4.34" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
  --quiet
!pip install --no-deps --force-reinstall \
  "flax==0.10.4" \
  "optax==0.2.5" \
  "chex==0.1.89" \
  "orbax-checkpoint==0.11.5" \
  --quiet

import jax, flax, optax
print("JAX:", jax.__version__, "Flax:", flax.__version__, "Optax:", optax.__version__)


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/kaggle/working/maxtext


/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
JAX: 0.4.34 Flax: 0.10.4 Optax: 0.2.5


## 4. Verify TPU Devices and Basic JAX Runtime
Quick sanity check: confirm 8 TPU devices and a trivial JAX op run. This ensures runtime is consistent before loading MaxText.


In [15]:
# 4) TPU device and trivial op
import jax, jax.numpy as jnp
n = jax.device_count()
print(f"TPU devices: {n}")
print("Trivial JAX op:", jnp.add(1, 2))
if n != 8:
    print("⚠️ Warning: Expected 8 TPU cores. Verify accelerator is TPU v3-8.")


TPU devices: 8
Trivial JAX op: 3


## 5. Configure Kaggle Dataset Checkpoint Path
Set the path to the uploaded Kaggle dataset containing the MaxText Orbax checkpoint and validate key files exist.
- Accept either `<slug>/llama-3.1-8b-maxtext-checkpoint/` or directly `<slug>/` structures.


In [16]:
# 5) Determine checkpoint directory in Kaggle input
from pathlib import Path

DATASET_SLUG = "llama-3-1-8b-maxtext-checkpoint"  # change to your dataset slug if different
ROOT = Path("/kaggle/input") / DATASET_SLUG

inner = ROOT / "llama-3.1-8b-maxtext-checkpoint"
if (inner / "_CHECKPOINT_METADATA").exists() or (inner / "0").exists():
    base = inner
else:
    base = ROOT

# prefer step dir "0" if exists, else last numeric
step = None
if (base / "0").exists():
    step = base / "0"
else:
    nums = [p for p in base.iterdir() if p.is_dir() and p.name.isdigit()]
    if nums:
        step = sorted(nums, key=lambda p: int(p.name))[-1]

CKPT_DIR = step if step else base
print("Dataset root:", ROOT)
print("Checkpoint dir:", CKPT_DIR)

required = [
    CKPT_DIR / "_CHECKPOINT_METADATA",
    CKPT_DIR / "items" / "_METADATA",
]
for p in required:
    print("Exists", p, p.exists())

items_dir = CKPT_DIR / "items"
print("Items dir:", items_dir, items_dir.exists())


Dataset root: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Checkpoint dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/_CHECKPOINT_METADATA True
Exists /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/_METADATA True
Items dir: /kaggle/input/llama-3-1-8b-maxtext-checkpoint/items True


## 6. Generate Minimal MaxText Config
Create a tiny YAML config with `load_parameters_path` pointing to the checkpoint and `steps: 1` for a minimal verification run.


In [17]:
# 6) Write minimal config YAML
import yaml
from pathlib import Path

CONFIG_DIR = Path("/kaggle/working/config")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "minimal_maxtext_config.yaml"

cfg = {
    "run_name": "llama31_8b_verify",
    "load_parameters_path": str(CKPT_DIR),
    "steps": 1,
    "dataset_type": "none",
}

with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Config:")
print(CONFIG_PATH.read_text())


Config:
run_name: llama31_8b_verify
load_parameters_path: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
steps: 1
dataset_type: none



## 7. [DEFINITIVE REVISION] Force Git State, Install, Clean, and Execute Atomically
This single cell forces a known-good git state, installs MaxText for reliable imports, aggressively cleans artifacts, verifies `attentions.py`, and immediately runs via module mode with a robust PYTHONPATH.

In [18]:
# 7) [DEFINITIVE REVISION] Forcing state, installing, cleaning, and executing in one atomic step
%cd /kaggle/working/maxtext

# 1. Force Git State: Check out the known pre-Pallas commit IMMEDIATELY before execution.
KNOWN_GOOD_COMMIT = "5a6580f3b1784d5aabca3291501e83548ab47235"
print(f"\U0001F529 Forcing git checkout to known-good commit: {KNOWN_GOOD_COMMIT[:10]}...")
!git checkout {KNOWN_GOOD_COMMIT} --force

# 2. Install Package: Properly install MaxText to make local modules importable.
print("\U0001F527 Installing MaxText in editable mode to resolve 'checkpointing'...")
!pip install -e . --quiet

# 3. Aggressive Cleanup: Clear all caches and build artifacts.
print("\U0001F9F9 Performing aggressive cleanup...")
!find . -type d -name "__pycache__" -exec rm -r {} +
!rm -rf build dist ./*.egg-info
print("✅ Environment prepared.")

# 4. Verification: Check the file content one last time.
print("\n\U0001F50D Verifying on-disk file content of attentions.py:")
!head -n 30 MaxText/layers/attentions.py | grep "pallas" || echo "✅ Pallas import not found, as expected."

# 5. Robust Execution: Run as a module with a fully specified PYTHONPATH.
print("\n\U0001F680 Attempting robust execution...")
import os, subprocess
entrypoint_module = "MaxText.train"
if os.path.exists("MaxText/train.py"):
    env = os.environ.copy()
    repo_root = "/kaggle/working/maxtext"
    package_root = "/kaggle/working/maxtext/MaxText"
    env['PYTHONPATH'] = f"{repo_root}:{package_root}:{env.get('PYTHONPATH','')}"

    cmd = ['python3', '-m', entrypoint_module, f'--config={CONFIG_PATH}']
    print("Running with updated PYTHONPATH:", " ".join(cmd))

    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=300, env=env)
        print("Return code:", res.returncode)
        if res.returncode == 0:
            print("✅✅✅ SUCCESS! Model loaded without error. ✅✅✅")
        print("\nSTDOUT (last 1200 chars):\n", res.stdout[-1200:])
        if res.stderr:
            print("\nSTDERR (last 800 chars):\n", res.stderr[-800:])
    except subprocess.TimeoutExpired:
        print("⏰ TIMEOUT: verification run exceeded 300s")
else:
    print("❌ Entrypoint not found.")


/kaggle/working/maxtext
🔩 Forcing git checkout to known-good commit: 5a6580f3b1...
Previous HEAD position was 44dceb02 refactor
HEAD is now at 5a6580f3 add grain instructions
🔧 Installing MaxText in editable mode to resolve 'checkpointing'...
ERROR: file:///kaggle/working/maxtext does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
🧹 Performing aggressive cleanup...
✅ Environment prepared.

🔍 Verifying on-disk file content of attentions.py:
from jax.experimental.pallas.ops import attention as pallas_attention
from jax.experimental.pallas.ops.tpu.splash_attention import splash_attention_mask
from jax.experimental.pallas.ops.tpu.splash_attention import splash_attention_kernel

🚀 Attempting robust execution...
Running with updated PYTHONPATH: python3 -m MaxText.train --config=/kaggle/working/config/minimal_maxtext_config.yaml
Return code: 1

STDO